In [1]:
# notebooks/main_training.ipynb

# Cell 1: Setup
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import os

# Import your modules
from configs.config import Config
from data.dataset import BraTSDataset
from models.bu_net_multitask import BUNetMultiTask
from training.trainer import Trainer

# Create directories
Config.create_dirs()

print(f"✓ Device: {Config.DEVICE}")
print(f"✓ Data path: {Config.TRAIN_DATASET_PATH}")

Verifying BraTS2020 dataset via KaggleHub...
✓ Device: mps
✓ Data path: /Users/yuzheli/.cache/kagglehub/datasets/awsaf49/brats20-dataset-training-validation/versions/1/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData


In [2]:
# Cell 2: Fix filename issue
old_name = os.path.join(Config.TRAIN_DATASET_PATH, 
                        "BraTS20_Training_355/W39_1998.09.19_Segm.nii")
new_name = os.path.join(Config.TRAIN_DATASET_PATH, 
                        "BraTS20_Training_355/BraTS20_Training_355_seg.nii")

if os.path.exists(old_name):
    os.rename(old_name, new_name)
    print("✓ Fixed BraTS20_Training_355 filename")
else:
    print("✓ Filename already correct")

✓ Filename already correct


In [3]:
# Cell 3: Create data splits
def get_patient_ids(data_path):
    dirs = [f.path for f in os.scandir(data_path) if f.is_dir()]
    return [d[d.rfind('/')+1:] for d in dirs]

all_ids = get_patient_ids(Config.TRAIN_DATASET_PATH)

# Split: 70% train, 20% val, 10% test
train_val_ids, test_ids = train_test_split(all_ids, test_size=0.1, random_state=42)
train_ids, val_ids = train_test_split(train_val_ids, test_size=0.22, random_state=42)  # 0.22 * 0.9 ≈ 0.2

print(f"✓ Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

✓ Train: 258, Val: 74, Test: 37


In [4]:
# Update Cell 4 in main_training.ipynb

from data.synthetic_tumor_generator import SyntheticAugmentedDataset

# INSTEAD OF:
train_dataset = BraTSDataset(train_ids, Config.TRAIN_DATASET_PATH)

# USE:
base_dataset = BraTSDataset(train_ids, Config.TRAIN_DATASET_PATH)
train_dataset = SyntheticAugmentedDataset(
    real_dataset=base_dataset,
    synthetic_ratio=0.3,  # 30% of samples will be synthetic
    img_size=Config.IMG_SIZE
)

# Val/test stay real-only
val_dataset = BraTSDataset(val_ids, Config.TRAIN_DATASET_PATH)
test_dataset = BraTSDataset(test_ids, Config.TRAIN_DATASET_PATH)

✓ Valid patients: 258/258
✓ Valid patients: 258/258
✓ Valid patients: 74/74
✓ Valid patients: 37/37


In [5]:
# Add this after creating the datasets in your notebook
def custom_collate(batch):
    """Custom collate function to handle dictionaries with mixed keys"""
    elem = batch[0]
    batch_dict = {}
    
    for key in elem.keys():
        if key == 'synthetic':
            # Handle boolean keys by converting to tensor
            batch_dict[key] = torch.tensor([d[key] for d in batch])
        else:
            # Default collate for other keys
            batch_dict[key] = torch.utils.data.dataloader.default_collate([d[key] for d in batch])
    
    return batch_dict

# Update your DataLoader creation:
train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,
    drop_last=True,
    collate_fn=custom_collate  # Add this line
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,
    collate_fn=custom_collate  # Add this line
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,
    collate_fn=custom_collate  # Add this line
)

In [6]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,
    drop_last=True  # Drop last incomplete batch
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=0
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=0
)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")

✓ Train batches: 3225
✓ Val batches: 925
✓ Test batches: 463


In [7]:
# Create model
print("Creating model...")
model = BUNetMultiTask(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
    num_tumor_types=Config.NUM_TUMOR_TYPES
).to(Config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model created: {Config.ENCODER_NAME}")
print(f"✓ Total parameters: {total_params:,}")
print(f"✓ Trainable parameters: {trainable_params:,}")

Creating model...
✓ Model created: resnet34
✓ Total parameters: 24,566,281
✓ Trainable parameters: 24,566,281


In [8]:
# Cell 6: Test forward pass
print("Testing forward pass...")
test_batch = next(iter(train_loader))
test_images = test_batch['image'][:2].to(Config.DEVICE)
test_masks = test_batch['mask'][:2].to(Config.DEVICE)

with torch.no_grad():
    seg_out, cls_out = model(test_images)
    
print(f"✓ Input: {test_images.shape}")
print(f"✓ Segmentation output: {seg_out.shape}")
print(f"✓ Classification output: {cls_out.shape}")
print(f"✓ Forward pass successful!")

Testing forward pass...
✓ Input: torch.Size([2, 2, 128, 128])
✓ Segmentation output: torch.Size([2, 4, 128, 128])
✓ Classification output: torch.Size([2, 5])
✓ Forward pass successful!


In [9]:
# Cell 7: Train model
print("="*60)
print("STARTING TRAINING")
print("="*60)

trainer = Trainer(
    model=model,
    config=Config,
    train_loader=train_loader,
    val_loader=val_loader
)

history = trainer.train()

STARTING TRAINING
Starting training on mps
Train batches: 3225, Val batches: 925


Epoch 1/35:  45%|████▌     | 1456/3225 [13:20<16:12,  1.82it/s, loss=0.3279, seg=0.3279, cls=0.0000]


KeyboardInterrupt: 

In [ ]:
# Cell 8: Visualize predictions
import matplotlib.pyplot as plt
import torch.nn.functional as F

def visualize_predictions(model, dataloader, num_samples=4):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    batch = next(iter(dataloader))
    images = batch['image'][:num_samples].to(Config.DEVICE)
    masks = batch['mask'][:num_samples]
    
    with torch.no_grad():
        seg_out, _ = model(images)
        predictions = torch.argmax(seg_out, dim=1).cpu()
    
    for i in range(num_samples):
        # Original FLAIR
        axes[i, 0].imshow(images[i, 0].cpu(), cmap='gray')
        axes[i, 0].set_title('FLAIR')
        axes[i, 0].axis('off')
        
        # Original T1CE
        axes[i, 1].imshow(images[i, 1].cpu(), cmap='gray')
        axes[i, 1].set_title('T1CE')
        axes[i, 1].axis('off')
        
        # Ground Truth
        axes[i, 2].imshow(masks[i], cmap='viridis')
        axes[i, 2].set_title('Ground Truth')
        axes[i, 2].axis('off')
        
        # Prediction
        axes[i, 3].imshow(predictions[i], cmap='viridis')
        axes[i, 3].set_title('Prediction')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, 'predictions.png'))
    plt.show()

visualize_predictions(model, val_loader, num_samples=4)